# E-Commerce Clickstream Analytics — Exploratory Data Analysis
**DATA 228 — Spring 2026 | Team 3**

This notebook provides interactive exploration of both datasets:
1. **REES46** eCommerce Behavior Data (~285M events)
2. **Criteo** 1TB Click Logs (~4.3B records, subset)

---

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

from config.settings import (
    spark_config, criteo_config, rees46_config,
    REES46_BRONZE_DIR, CRITEO_BRONZE_DIR
)

spark = SparkSession.builder \
    .appName('EDA_Notebook') \
    .master('local[*]') \
    .config('spark.driver.memory', '8g') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()

print(f'Spark version: {spark.version}')

## 1. REES46 eCommerce Behavior Data

In [ ]:
# Load Bronze data
events = spark.read.parquet(REES46_BRONZE_DIR)
print(f'Total events: {events.count():,}')
print(f'Schema:')
events.printSchema()

In [ ]:
# Sample data
events.show(10, truncate=False)

In [ ]:
# ─── Event Type Distribution ───
event_dist = events.groupBy('event_type').agg(
    count('*').alias('count'),
    countDistinct('user_id').alias('unique_users'),
    countDistinct('product_id').alias('unique_products')
).toPandas()

fig = px.pie(event_dist, values='count', names='event_type',
             title='Event Type Distribution', hole=0.4,
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.show()

In [ ]:
# ─── Daily Event Volume ───
daily = events.groupBy('event_date').agg(
    count('*').alias('total_events'),
    countDistinct('user_id').alias('unique_users'),
    sum(when(col('event_type') == 'purchase', col('price')).otherwise(0)).alias('revenue')
).orderBy('event_date').toPandas()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Daily Events', 'Daily Revenue'])
fig.add_trace(go.Scatter(x=daily['event_date'], y=daily['total_events'],
                         name='Events', fill='tozeroy'), row=1, col=1)
fig.add_trace(go.Scatter(x=daily['event_date'], y=daily['revenue'],
                         name='Revenue ($)', fill='tozeroy',
                         line=dict(color='#00CC96')), row=2, col=1)
fig.update_layout(height=600, title='Daily Trends')
fig.show()

In [ ]:
# ─── Hourly Activity Pattern ───
hourly = events.withColumn('hour', hour('event_time')) \
    .groupBy('hour', 'event_type').agg(count('*').alias('count')) \
    .orderBy('hour').toPandas()

fig = px.line(hourly, x='hour', y='count', color='event_type',
              title='Hourly Activity by Event Type',
              labels={'hour': 'Hour of Day', 'count': 'Event Count'})
fig.show()

In [ ]:
# ─── Top Categories ───
top_cats = events.filter(col('event_type') == 'purchase') \
    .groupBy('main_category').agg(
        count('*').alias('purchases'),
        sum('price').alias('revenue'),
        countDistinct('user_id').alias('buyers')
    ).orderBy(col('revenue').desc()).limit(15).toPandas()

fig = px.bar(top_cats, x='main_category', y='revenue',
             color='purchases', title='Top 15 Categories by Revenue',
             color_continuous_scale='Viridis')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# ─── Price Distribution ───
prices = events.filter(
    (col('price') > 0) & (col('price') < 1000)
).select('price', 'event_type').sample(0.01).toPandas()

fig = px.histogram(prices, x='price', color='event_type', nbins=100,
                   title='Price Distribution by Event Type (sample)',
                   marginal='box', opacity=0.7)
fig.show()

In [ ]:
# ─── Conversion Funnel ───
funnel = events.groupBy('event_type').agg(
    count('*').alias('total'),
    countDistinct('user_id').alias('unique_users')
).toPandas()

order = ['view', 'cart', 'purchase']
funnel['event_type'] = pd.Categorical(funnel['event_type'], categories=order, ordered=True)
funnel = funnel.sort_values('event_type')

fig = go.Figure(go.Funnel(
    y=funnel['event_type'], x=funnel['unique_users'],
    textinfo='value+percent initial',
    marker=dict(color=['#636EFA', '#FFA15A', '#00CC96'])
))
fig.update_layout(title='Conversion Funnel (Unique Users)', height=350)
fig.show()

In [ ]:
# ─── User Activity Distribution ───
user_activity = events.groupBy('user_id').agg(
    count('*').alias('total_events'),
    countDistinct('event_date').alias('active_days'),
    sum(when(col('event_type') == 'purchase', 1).otherwise(0)).alias('purchases')
).toPandas()

fig = px.histogram(user_activity, x='total_events', nbins=100,
                   title='User Activity Distribution (events per user)',
                   log_y=True, labels={'total_events': 'Events per User'})
fig.show()

print(f"Users with purchases: {(user_activity['purchases'] > 0).sum():,} / {len(user_activity):,}")
print(f"Conversion rate: {(user_activity['purchases'] > 0).mean():.2%}")

In [ ]:
# ─── Brand Analysis ───
top_brands = events.filter(col('brand') != 'unknown') \
    .groupBy('brand').agg(
        count('*').alias('interactions'),
        sum(when(col('event_type') == 'purchase', col('price')).otherwise(0)).alias('revenue'),
        countDistinct('user_id').alias('users')
    ).orderBy(col('revenue').desc()).limit(20).toPandas()

fig = px.treemap(top_brands, path=['brand'], values='revenue',
                 color='users', color_continuous_scale='RdYlGn',
                 title='Top 20 Brands by Revenue')
fig.show()

## 2. Criteo Click Logs

In [ ]:
# Load Criteo Bronze data
criteo = spark.read.parquet(CRITEO_BRONZE_DIR)
print(f'Total records: {criteo.count():,}')
criteo.printSchema()

In [ ]:
# ─── Label Distribution (CTR) ───
label_dist = criteo.groupBy('label').count().toPandas()
total = label_dist['count'].sum()
label_dist['percentage'] = label_dist['count'] / total * 100

fig = px.bar(label_dist, x='label', y='count', color='label',
             text='percentage', title='Click vs No-Click Distribution')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.show()

ctr = label_dist[label_dist['label'] == 1]['percentage'].values[0]
print(f'Overall CTR: {ctr:.2f}%')

In [ ]:
# ─── Integer Feature Distributions ───
int_cols = criteo_config.int_feature_cols
sample_criteo = criteo.select(int_cols).sample(0.001).toPandas()

fig = make_subplots(rows=3, cols=5, subplot_titles=int_cols[:15])
for i, c in enumerate(int_cols[:15]):
    row, col_idx = (i // 5) + 1, (i % 5) + 1
    fig.add_trace(
        go.Histogram(x=sample_criteo[c].dropna(), nbinsx=50, name=c,
                     showlegend=False),
        row=row, col=col_idx
    )
fig.update_layout(height=600, title='Integer Feature Distributions (sample)')
fig.show()

In [ ]:
# ─── Missing Value Analysis ───
null_counts = []
for c in int_cols:
    null_pct = criteo.filter(col(c) == 0.0).count() / criteo.count() * 100
    null_counts.append({'feature': c, 'null_pct': null_pct})

null_df = pd.DataFrame(null_counts)
fig = px.bar(null_df, x='feature', y='null_pct',
             title='Missing Value % per Integer Feature',
             labels={'null_pct': 'Missing %'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# ─── Feature Correlation with Label ───
corr_data = []
for c in int_cols:
    corr = criteo.select(corr(c, 'label')).first()[0]
    corr_data.append({'feature': c, 'correlation': corr or 0})

corr_df = pd.DataFrame(corr_data).sort_values('correlation', key=abs, ascending=False)
fig = px.bar(corr_df, x='feature', y='correlation',
             color='correlation', color_continuous_scale='RdBu_r',
             title='Feature Correlation with Click Label')
fig.show()

## 3. Summary Statistics

In [ ]:
# ─── Final Summary ───
print('=' * 60)
print('  DATASET SUMMARY')
print('=' * 60)
print(f'  REES46 Events:       {events.count():>15,}')
print(f'  REES46 Users:        {events.select(countDistinct("user_id")).first()[0]:>15,}')
print(f'  REES46 Products:     {events.select(countDistinct("product_id")).first()[0]:>15,}')
print(f'  Criteo Records:      {criteo.count():>15,}')
print(f'  Criteo CTR:          {ctr:>14.2f}%')
print('=' * 60)

In [ ]:
spark.stop()